# Producing events with Apache Kafka running on Confluent

For our course we'll be using Kafka hosted on Confluent Cloud, which offers a managed Kafka Service.
Now in order to be able to connect to the managed kafka service we'll need following packages, which are already installed in this github codespace environment


```python
pip install confluent-kafka[avro,json,protobuf]>=1.4.2
```

We'll be mainly using the library, `confluent_kafka` which provides all necessary functionalities to allow interaction with a kafka service. It does contain a confluent_kafka `Producer`and `Consumer` class, we'll be using in our session.

For futher details and functionalities offered by the confluent_kafka library you can check out the documentation. https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html

## Helper module ccloud_lib.py 
We have included a helper module inside utils, called `ccloud_lib.py` (confluent cloud lib) this module wrapps some functionality for us, making the creating of a kafka app more handy.

In [ ]:
import utils.ccloud_lib as ccloud_lib

### Get connection details and read into dict
Our confluent server authentication details can be found inside the `kafka.config` file. 

In [ ]:
# see what's inside the kafka.config file
! cat kafka.config

The ccloud_lib has a function that allows parsing the kafka config credentials and reading them into a dictionary for later use.

In [ ]:
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")
kafka_app_config

### Kafka Topic

As you have already learned, a Kafka topic is a category or feed name to which records are published. 

* Topics are always multi-subscriber; that is, a topic can have zero, one, or many consumers that subscribe to the data written to it. 
* Topics are divided into partitions, allowing them to be scaled horizontally, and each partition is an ordered, immutable sequence of records that is continually appended to—a structured commit log. 


Kafka topics are a core component in the management and organization of data within Kafka and serve as the routing mechanism for record publication and consumption.

<img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/0*zGVxdZ8z4-KZL9qM.png" style="width: 70%">

#### Name your kafka topic
Now in order to create our custom topic, we'll have to provide a name for it. 

In [ ]:
# use your student-id for instance
topic_name = "YOUR-TOPIC-NAME"

#### Create the topic using the AdminClient

You can create a topic using the AdminClient. The AdminClient is a client that allows you to manage and inspect topics, brokers, and configurations. 
It returns dict of futures for each topic, keyed by the topic name.
Consumers and producers can only ever read/write to one partition. Replicas only help when brokers are down.

In [ ]:
### Create a topic
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient(kafka_app_config)
# Call create_topics to create topics. A dict of <topic,future> is returned.
# replication_factors are optional. If not supplied, the default is used which is the cluster's replication factor.
# A replication is a copy of the topic's data. If a broker is down, another broker can serve the data.
futures_result = admin_client.create_topics([NewTopic(topic_name, num_partitions=1, replication_factor=3)])
display(futures_result)

### Wait for the creation of the topic
we can check if the topic was created successfully by waiting for the futures to complete:

In [ ]:
for topic, f in futures_result.items():
    f.result()  # The result itself is None
    print("Topic {} created".format(topic))

### Creating our producer
A Producer is an instance that sends messages to Kafka topics. It is thread-safe and can be shared among many threads. 
Now that we have a topic, we can create a producer that sends messages to that topic. It is really simple to create a producer in Python using the Confluent Kafka Python client.


1. **Import the confluent_kafka producer**

In [ ]:
# first we import the KafkaProducer
from confluent_kafka import Producer

2. **Create a producer object**

In [ ]:
# then we create a producer object
producer = Producer(kafka_app_config)

3. **Produce messages to the topic:** A message consists of a key and a value that the producer sends to a Kafka topic. The key is used to determine the partition within the topic where the record will be sent, and the value is the actual message payload. Optionally, a timestamp and headers can also be included with each message. The `produce()` method is used to send messages to the Kafka topic. It takes the topic name, the message key, and value, and optionally a callback function. The key helps Kafka to maintain the order of messages within a single partition.


In [ ]:
import json

message = {'message': f"Hello, from {topic_name}!"}
message_id = "1"
producer.produce(topic_name, key=message_id, value=json.dumps(message))

4. **Flush and poll** the producer to ensure all messages are delivered. 

    **The poll()** method on the Kafka producer is primarily used to invoke the delivery report callback that is set when sending a message using the produce() method. The argument inside poll() represents the maximum amount of time in milliseconds that poll() will block waiting for events.
    
    **producer.flush()**
    This method is used to ensure that all messages sent by the producer are fully completed and acknowledged by the Kafka broker(s). This is crucial when you need to guarantee that no messages are left unsent before your application shuts down or transitions to another phase of operation.

In [ ]:
producer.poll(1) # Wait for any outstanding messages to be delivered and report delivery results
producer.flush()  # Important to flush to make sure all messages are sent

5. **Deliver with delivery callback**

    You can also produce messages with a delivery callback to check if the message was sent successfully: 

In [ ]:
def delivery_report(err, msg):
    if err is not None:
        print(f'Message delivery failed: {err}')
    else:
        print(f'Message delivered to {msg.topic()} [{msg.partition()}]')    

In [ ]:
message = {'message': f"Hello, from {topic_name}!"}
message_id = "1"
producer.produce(topic_name, key=message_id, value=json.dumps(message), callback=delivery_report)
producer.flush()  # Important to flush to make sure all messages are sent
producer.poll(0)

#### Messages where produced
We can now consume the message from the topic we created. We will use the KafkaConsumer for this. But first let's have a look inside the confluent cloud UI to see if the message was produced.

Run the production of messages a few times, so we have multiple events generated!

##### (Disclaimer Understanding Kafka Producer Acknowledgment Settings `acks`)
The default configuration for acks in Kafka producers is 1, where the producer waits for the leader broker only to acknowledge the message.​ The Kafka producer configuration for acks can be set in the config dictionary passed during instantiation

```python
config = {
    'bootstrap.servers': ....
    ....
    'acks': 'all'  # Setting acks to 'all' for full acknowledgment
}
producer = Producer(config)
```

**ACKS parameter**

The `acks` parameter in Kafka dictates how producers receive confirmations from brokers about message delivery, influencing reliability and performance. Below is an example of setting `acks` in the producer configuration, followed by a table summarizing the possible settings.

| `acks` Value | Description                                                                                       |
|--------------|---------------------------------------------------------------------------------------------------|
| `0`          | No acknowledgments. Producer considers a message sent immediately after sending.                  |
| `1`          | Leader acknowledgment. Producer waits for the leader broker only to acknowledge the message.     |
| `all`        | Full acknowledgment. Producer waits for all in-sync replicas (ISRs) to acknowledge the message.   |

## Consuming messages
Now when it comes down to consuming messages, we will use the KafkaConsumer class. This class allows us to subscribe to a topic and poll for new messages. We will also use the json module to parse the message value.

In order to consume message, we'll need to specify a **group.id**, so we can uniquely identify our consumer.

In [ ]:
group_id = "YOUR STUDENT / GROUP ID"
kafka_app_config["group.id"] = group_id

### Explanation of `group.id` in Kafka

- **Consumer Group Management**:
  The `group.id` uniquely identifies a consumer group in Kafka, which is a collection of consumer instances working together to consume and process records from one or more topics efficiently.

- **Load Balancing**:
  By utilizing `group.id`, Kafka can evenly distribute topic partitions across the consumers in a group. This allows for parallel processing and improves both scalability and fault tolerance.

- **Fault Tolerance**:
  When a consumer fails, Kafka uses the `group.id` to reassign the partitions of the failed consumer to other active consumers in the same group. This mechanism ensures that there is no interruption in data processing and no data loss.

- **Offset Management**:
  Kafka tracks the progress of each consumer in terms of offsets, which are pointers to the last record read from a partition. The `group.id` allows Kafka to manage offsets at the group level. This way, if a consumer stops or fails, other consumers in the group can resume reading from where the last consumer left off, ensuring continuity and avoiding reprocessing of data.

### Practical Use Cases of `group.id`

- **Multiple Consumer Instances**:
  Inside a consumer group, multiple instances can coordinate the consumption of partitions. Each consumer is assigned a set of partitions, and no two consumers in the same group read from the same partition simultaneously.

- **Rebalancing**:
  Consumer groups experience a rebalance process when members of the group change (additions or removals), or when the partitions of subscribed topics change. Rebalancing involves reassigning the partition ownership among the consumers based on the `group.id`, ensuring efficient data consumption and processing.

<img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*ApW4J5g-q8zeJ8aMiSYRJw.png" alt="drawing" width="75%"/>

#### Specifying the reading mode:
Now we will specify the reading mode. The reading mode will specify to the consumer where to start reading the messages from. We want to read from the beginning of the topic, so we will set the reading mode to "earliest".

In [ ]:
kafka_app_config["auto.offset.reset"] = "earliest"


##### Options for `auto.offset.reset`

The `auto.offset.reset` configuration determines how a Kafka consumer behaves when it finds no initial valid offsets or when the offsets have expired. This setting is critical for managing data continuity and integrity in your Kafka streams. Below is a detailed description of each option:

| Setting    | Description |
|------------|-------------|
| `earliest` | This option configures the consumer to start reading at the earliest offset possible when no previous offsets are found for the consumer's group in the offset store. This is useful for ensuring that no messages are missed when a new consumer group is created or after offsets have expired. However, this could lead to reprocessing messages if not carefully managed. |
| `latest`   | When set to `latest`, the consumer will start reading from the newest records, ignoring any messages that were published before the consumer was started. This setting is most useful for applications that only care about real-time data processing and can safely ignore historical data. |
| `none`     | If `none` is selected, the consumer will throw an exception if it encounters a situation where no valid offsets are found and no default behavior is defined. This setting is suitable for scenarios where you need to ensure strict processing discipline, as it forces the handling of offset management and recovery procedures explicitly. |

### Choosing the Right Setting for `auto.offset.reset`

Selecting the appropriate `auto.offset.reset` setting depends on the specific requirements and characteristics of your application:

- **Batch processing or historical data analysis**: Choose `earliest` if your application needs to process all available data from a topic from the very beginning.
- **Real-time data processing applications**: Select `latest` if your application should only process messages from the point it starts running onwards, ignoring any old data.
- **High-reliability systems where data integrity is critical**: Use `none` to ensure that any potential issues with offsets are immediately flagged and can be addressed manually or through automated recovery mechanisms.

It's important to align the `auto.offset.reset` configuration with your data processing goals and operational policies to maximize the effectiveness and reliability of your Kafka consumer applications.

In [ ]:
kafka_app_config['group.id'] = group_id
kafka_app_config['auto.offset.reset'] = 'earliest'

#### Create consumer Object and subscribe to topic
Now we create a consumer with the configuration we created and we subscribe to the topic we created

In [ ]:
from confluent_kafka import Consumer

# create a consumer object
consumer = Consumer(kafka_app_config)
consumer.subscribe([topic_name])

### Processing Kafka Messages with a Consumer
The following code defines how a Kafka consumer processes messages it retrieves. The run_consumer method continuously polls the Kafka topic for messages, handles errors, and processes the received data.

In [ ]:
import json
def run_consumer(consumer):
    while True:
        msg = consumer.poll(0.5)
        if msg is None:
            # No message available within timeout.
            # Initial message consumption may take up to
            # `session.timeout.ms` for the consumer group to
            # rebalance and start consuming
            print("Waiting for message or event/error in poll()", end="\r", flush=True)
            continue
        elif msg.error():
            print('error: {}'.format(msg.error()))
        else:
            # Check for Kafka message
            record_key = msg.key()
            record_value = msg.value()
            data = json.loads(record_value)
            print(data)

### Create consumer Object and subscribe to topic
Now we create a consumer with the configuration we created and we subscribe to the topic we created

In [ ]:
consumer = Consumer(kafka_app_config)
consumer.subscribe([topic_name])
run_consumer(consumer)

In [ ]:
consumer.close()

#### Some new messages 
Produce some new messages and run your consumer again, to see, that only new messages will arrive now

In [ ]:
## Produce some messages again:

producer.produce(topic_name, key=message_id, value=json.dumps(message), callback=delivery_report)
producer.produce(topic_name, key="12", value=json.dumps({'message': f"Another, one time mega hello, from {topic_name}!"}), callback=delivery_report)
producer.produce(topic_name, key="31", value=json.dumps({'message': f"One more, hello! from {topic_name}!"}), callback=delivery_report)
producer.produce(topic_name, key="44", value=json.dumps({'message': f"Hola! from {topic_name}!"}), callback=delivery_report)
producer.flush()  # Important to flush to make sure all messages are sent
producer.poll(0)

##### Run the consumer again

In [ ]:
kafka_app_config['auto.offset.reset'] = 'earliest'
consumer = Consumer(kafka_app_config)
consumer.subscribe([topic_name])
run_consumer(consumer)

In [ ]:
consumer.close()

#### Understanding Consumer Behavior with Different `group.id`
When a Kafka consumer changes its `group.id`, it is essentially treated as a new consumer group by the Kafka brokers. This has implications for message consumption, particularly in relation to where the consumption starts based on the `auto.offset.reset` setting. 

In [ ]:
# Configuring the consumer to read from the beginning of the topic
kafka_app_config['group.id'] = 'NEW GROUP ID'
kafka_app_config['auto.offset.reset'] = 'earliest'

# Create a new consumer object with the updated configuration
consumer = Consumer(kafka_app_config)
consumer.subscribe([topic_name])
run_consumer(consumer)

**Again, Please make sure to stop the consumer after you are done with it.**

In [ ]:
consumer.close()

## Resetting Consumer Offsets to the Earliest

When managing Kafka consumers, you might encounter situations where you need precise control over which partitions to consume from and at what offset to start. Direct partition assignment bypasses the Kafka consumer group's coordination, allowing you to explicitly specify partition numbers and starting offsets. 

```python
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer(kafka_app_config)
tp = TopicPartition(topic=topic_name, partition=0, offset=0)
consumer.assign([tp])

run_consumer(consumer)

In [ ]:
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer(kafka_app_config)
tp = TopicPartition(topic=topic_name, partition=0, offset=0)
consumer.assign([tp])

run_consumer(consumer)

In [ ]:
consumer.close()

### Consume music event streams

We can also access and display all streams coming in from our music_streams topic. Simply set the topic_name to music_streams and run the consumer.

In [ ]:
from confluent_kafka import Consumer

kafka_app_config['group.id'] = 'NEW GROUP ID'
kafka_app_config['auto.offset.reset'] = 'latest'

consumer = Consumer(kafka_app_config)
consumer.subscribe(['music_streams'])
run_consumer(consumer)

In [ ]:
consumer.close()

### Do something more meaningful with the data
We can also do something more meaningful with the data. For example, we can count the songplays per user and show that information

In [ ]:
# Dictionary to keep track of song plays per user
song_plays = {}

def run_song_plays(consumer):
    while True:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            print("Consumer error: {}".format(msg.error()))
        else:
            # Proper message
            process_message(msg.value().decode('utf-8'))
    
def process_message(message):
    try:
        event = json.loads(message)
        user_id = event.get('userId')
        if user_id:
            song_plays[user_id] = song_plays.get(user_id, 0) + 1
            print(f'User {user_id} has listened to {song_plays[user_id]} songs.')
    except json.JSONDecodeError:
        print("Failed to decode JSON message")

In [ ]:
# Run the consumer

kafka_app_config['group.id'] = 'NEW GROUP ID'
# Start reading at the beginning of the topic - so we have few more counts
kafka_app_config['auto.offset.reset'] = 'earliest' 
consumer = Consumer(kafka_app_config)
consumer.subscribe(['music_streams'])
run_song_plays(consumer)

In [ ]:
consumer.close()